# Kien Giang spatial-model experiments

This notebook preserves preprocessing and regression attempts from revision `19430e7`, with no successful estimates or model figures saved.
Eugenia's final panel and instrumental-variable analysis uses her Stata workflow.

## Setup and inputs

Requires `pandas`, `numpy`, `dask[dataframe,distributed]`, `psutil`, `pyarrow`, `statsmodels`, `scipy`, `spreg`, `libpysal`, and `dask-glm`.
Exact versions were not recorded.
The first cell starts five Dask workers with 8 GB each; review available memory before execution.

| Path under `D:/Work/ADB/SAR/datasets` | Role |
|---|---|
| `SAR_SVN_rice_KienGiang_28Feb2025.csv` | Province input |
| `KienGiang_rice_processed_optimized` | Preprocessing output and model input |
| `KienGiang_rice_processed` | Separate Parquet branch for the driver and memory comparison |

Review paths and reconcile the two branches' rows/schemas before comparison.
Preprocessing writes Parquet; later fits load large arrays.
It selects reproductive/vegetative rows, imputes covariates, creates logs, and builds time/plot dummies.

## Execution status

Fitting alternatives remain unfinished; annotations describe their historical failures.
Cleanup retained memory summaries, removed tracebacks/logs, and checked schema and transformed Python syntax without running analysis or loading large inputs.
Resolve the model issues and memory requirements below before running `python -m jupyter nbconvert --execute --to notebook --inplace spatial_model.ipynb`.


## 1. Preprocessing

### Start Dask

Creates a cluster and suppresses warnings; inspect worker logs when troubleshooting.
Close sessions with `client.close()` and `cluster.close()`.


In [ ]:
import logging
import warnings
warnings.filterwarnings('ignore')
import gc
from datetime import datetime
import os
import psutil

import pandas as pd
import numpy as np
from dask.distributed import Client, LocalCluster
cluster = LocalCluster(n_workers=5, memory_limit="8GB", 
                       processes=True)
client = Client(cluster)
import dask.dataframe as dd

### Measure memory

Measures the current Python process, excluding separate Dask workers.


In [ ]:
# Suppress warnings
warnings.filterwarnings('ignore')

# Memory monitoring function
def print_memory_usage():
    process = psutil.Process(os.getpid())
    mem_gb = process.memory_info().rss / 1024 / 1024 / 1024
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Current memory usage: {mem_gb:.2f} GB")
    return mem_gb

### Set paths

Preprocessing writes `output_path`, not the separate `output_dir` branch.


In [ ]:
file_path = r"D:\Work\ADB\SAR\datasets\SAR_SVN_rice_KienGiang_28Feb2025.csv"
output_path = r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed_optimized"
output_dir = r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed/"

### Prepare the panel

Drops missing coordinates/NDVI and mean-imputes five covariates.
Clips four covariates at `0.0001` before logging, adds `0.01` to lagged nightlights before logging, and squares log-PM2.5.
Plot IDs round coordinates to three decimals, potentially combining nearby locations.
Above 1,000 plot IDs, only the most frequent 1,000 receive dummies.
These experimental rules differ from the manuscript sample.

Check dummies because delayed lambdas capture loop variables and branches omit different plot categories.
Exports downcast numbers and replace hyphens/decimal points in column names with underscores, affecting precision and selection.


In [ ]:
# Define necessary columns
needed_cols = ['lon', 'lat', 'ndvi', 'pm25_mean', 'temp_av', 'rain_cum', 
               'hum_av', 'lag_lumen', 'reprod', 'veg', 'year', 'mon']

print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Starting distributed data loading from {file_path}")
start_time = datetime.now()

# Read CSV with Dask - increase blocksize to reduce partitions
ddf = dd.read_csv(file_path, usecols=needed_cols, blocksize="100MB")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Created Dask DataFrame with {len(ddf.divisions)-1} partitions")

# Filter data to keep only reproductive or vegetative phase
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Filtering data for reproductive or vegetative phase")
ddf = ddf[(ddf['reprod'] == 1) | (ddf['veg'] == 1)]

# Handle missing values
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Handling missing values")
# First, compute statistics about missing values
missing_stats = ddf.isnull().sum().compute()
for col in ddf.columns:
    missing = missing_stats[col]
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Column {col}: {missing:,} missing values")

# Drop rows with missing values in critical columns
ddf = ddf.dropna(subset=['lon', 'lat', 'ndvi'])

# For other columns, fill with appropriate values
# We need to compute means first
numeric_cols = ['pm25_mean', 'temp_av', 'rain_cum', 'hum_av', 'lag_lumen']
means = ddf[numeric_cols].mean().compute()

for col in numeric_cols:
    fill_value = means[col]
    ddf[col] = ddf[col].fillna(fill_value)
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Filled {col} missing values with mean: {fill_value:.4f}")

# Create log transformations efficiently
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Creating log transformations of numeric variables")
for col in numeric_cols[:-1]:  # Excluding lag_lumen
    # Get min, max, and count of zeros
    stats = ddf[col].describe().compute()
    min_val = stats['min']
    max_val = stats['max']
    n_zeros = (ddf[col] == 0).sum().compute()
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {col} range: {min_val:.4f} to {max_val:.4f}, Zero values: {n_zeros:,}")
    
    # Add small constant to avoid log(0)
    ddf[f'ln_{col}'] = ddf[col].clip(lower=0.0001).map_partitions(np.log)

# Handle lag_lumen separately
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Processing lag_lumen")
lag_lumen_stats = ddf['lag_lumen'].describe().compute()
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - lag_lumen range before adjustment: {lag_lumen_stats['min']:.4f} to {lag_lumen_stats['max']:.4f}")
ddf['lag_lumen'] = ddf['lag_lumen'] + 0.01
ddf['ln_lag_lumen'] = ddf['lag_lumen'].map_partitions(np.log)

# Create squared term for log PM2.5
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Creating squared term for log PM2.5")
ddf['ln_pm25_mean_sq'] = ddf['ln_pm25_mean'] ** 2

# Create time identifier
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Creating time identifiers")
ddf['time_id'] = ddf['year'].astype(str) + '-' + ddf['mon'].astype(str).map_partitions(lambda s: s.str.zfill(2))

# We need to compute time_periods for later use
time_periods = ddf['time_id'].unique().compute()
time_periods = sorted(time_periods)
n_periods = len(time_periods)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Found {n_periods} unique time periods")

# Create plot ID from coordinates for fixed effects
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Creating plot ID from coordinates")
ddf['plot_id'] = ddf['lon'].round(3).astype(str) + '_' + ddf['lat'].round(3).astype(str)

# To avoid recursion depth issues, we'll persist the dataframe at this point
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Persisting DataFrame to optimize computation")
ddf = ddf.persist()

# For time period dummies, we can create them directly
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Creating time period dummy variables")
# Create a separate dataframe for time dummies to avoid recursion issues
time_dummies = []
for period in time_periods[1:]:  # Skip first period to avoid perfect multicollinearity
    time_dummy = ddf['time_id'].map(lambda x: 1 if x == period else 0, 
                                  meta=(f'time_{period}', 'int64'))
    time_dummies.append(time_dummy.rename(f'time_{period}'))

# Combine time dummies with main dataframe
if time_dummies:
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Combining time dummies with main dataframe")
    # Create a DataFrame from the time dummies
    time_df = dd.concat(time_dummies, axis=1)
    # Persist to avoid recomputation
    time_df = time_df.persist()
    # Join with main dataframe
    ddf = dd.concat([ddf, time_df], axis=1)
    # Persist again
    ddf = ddf.persist()
    del time_df
    gc.collect()

# For plot dummies, we need a different approach due to potential memory issues
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Identifying most frequent plots for dummy variables")
# Force computation of plot counts in a separate step
plot_counts = ddf['plot_id'].value_counts().compute()
n_plots = len(plot_counts)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Found {n_plots} unique plot IDs")

# If there are too many plots, use only the most frequent ones
if n_plots > 1000:
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Too many plot IDs ({n_plots}) for dummy variables. Using top 1000 most frequent plots.")
    top_plots = plot_counts.nlargest(1000).index.tolist()
    
    # Create plot dummies in batches to avoid recursion depth issues
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Creating plot dummies in batches")
    batch_size = 100  # Process this many plots at a time
    
    for i in range(0, len(top_plots), batch_size):
        batch_plots = top_plots[i:i+batch_size]
        print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Processing plot batch {i//batch_size + 1} of {(len(top_plots) + batch_size - 1)//batch_size}")
        
        plot_dummies = []
        for plot in batch_plots:
            dummy = ddf['plot_id'].map(lambda x: 1 if x == plot else 0, 
                                     meta=(f'plot_{plot}', 'int64'))
            plot_dummies.append(dummy.rename(f'plot_{plot}'))
        
        if plot_dummies:
            # Create a DataFrame from this batch of plot dummies
            batch_df = dd.concat(plot_dummies, axis=1)
            batch_df = batch_df.persist()
            # Join with main dataframe
            ddf = dd.concat([ddf, batch_df], axis=1)
            ddf = ddf.persist()
            del batch_df
            gc.collect()
else:
    # If number of plots is manageable, create dummies in batches
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Creating dummy variables for all plots in batches")
    plot_categories = plot_counts.index.tolist()[1:]  # Skip first to avoid multicollinearity
    batch_size = 100  # Process this many plots at a time
    
    for i in range(0, len(plot_categories), batch_size):
        batch_plots = plot_categories[i:i+batch_size]
        print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Processing plot batch {i//batch_size + 1} of {(len(plot_categories) + batch_size - 1)//batch_size}")
        
        plot_dummies = []
        for plot in batch_plots:
            dummy = ddf['plot_id'].map(lambda x: 1 if x == plot else 0, 
                                    meta=(f'plot_{plot}', 'int64'))
            plot_dummies.append(dummy.rename(f'plot_{plot}'))
        
        if plot_dummies:
            # Create a DataFrame from this batch of plot dummies
            batch_df = dd.concat(plot_dummies, axis=1)
            batch_df = batch_df.persist()
            # Join with main dataframe
            ddf = dd.concat([ddf, batch_df], axis=1)
            ddf = ddf.persist()
            del batch_df
            gc.collect()

# # Compute memory usage per partition instead of globally
# print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Estimating memory usage by partition")
# memory_usage_per_partition = ddf.map_partitions(lambda df: df.memory_usage(deep=True).sum()).compute()
# estimated_size_gb = memory_usage_per_partition.sum() / 1e9

# # Adjust partition count dynamically based on memory estimate
# partition_count = max(1, int(estimated_size_gb * 10))  # ~100MB per partition
# ddf = ddf.repartition(npartitions=min(partition_count, 100))
# ddf = ddf.persist()
# print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Estimated total size: {estimated_size_gb:.2f} GB, using {partition_count} partitions")

def downcast_types(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')  # Convert float64 → float32
    for col in df.select_dtypes(include=['int64']).columns:
        if col in ['year', 'plot_id']:
            df[col] = df[col].astype('int32')  # Convert int64 → int8
        else:
            df[col] = pd.to_numeric(df[col], downcast='integer')  # Convert to the smallest integer type
    # for col in df.select_dtypes(include=['object']).columns:
    #     df[col] = df[col].astype('category')  # Convert object to category
    return df

ddf_optimized = ddf.map_partitions(downcast_types)

ddf_optimized.columns = [col.replace('-', '_') for col in ddf_optimized.columns]
ddf_optimized.columns = [col.replace('.', '_') for col in ddf_optimized.columns]

ddf_optimized = ddf_optimized.persist()

# Write to Parquet
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Writing processed data to {output_path}")
try:
    ddf_optimized.to_parquet(output_path, compression="snappy", write_index=False)
except RecursionError:
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Encountered RecursionError, saving in chunks")
    
    # Save each partition separately
    os.makedirs(output_path.replace('.parquet', ''), exist_ok=True)
    
    for i, partition in enumerate(ddf_optimized.to_delayed()):
        df_part = partition.compute()
        df_part.to_parquet(f"{output_path.replace('.parquet', '')}/part-{i:05d}.parquet", compression="snappy", index=False)
        del df_part
        gc.collect()

end_time = datetime.now()
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Processing completed in {end_time - start_time}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Data saved successfully")

## 2. Chunk-level model experiment

### Fit chunk-level models

`process_chunk` fits OLS with HC3 and available time/plot dummies.
Spatial lag/error fits sample at most 3,000 rows with seed 42 and use six covariates without dummies.
Weights are row-standardized nearest neighbors in longitude/latitude, with nonmetric distances and potentially repeated locations.

No successful result is saved.
Check the nested multiprocessing worker and `spreg` result attributes before enabling the driver.
Prefix selection includes `time_id`/`plot_id` unless removed.


In [ ]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import sparse
from spreg import ML_Lag, ML_Error
from libpysal import weights
import logging
from datetime import datetime
import multiprocessing as mp
import glob
import scipy.stats as stats
pd.set_option('display.max_columns', 500)
import warnings
warnings.filterwarnings('ignore')

# Initialize logger
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

# Path to Parquet files
PARQUET_PATH = r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed/*.parquet"

# Sample size for spatial models
SAMPLE_SIZE = 3000

def process_chunk(df):
    """Processes a chunk of data and runs regressions efficiently."""
    
    logger.info(f"Processing chunk with {len(df):,} observations")
    
    try:
        # Convert dependent variable
        y = df['ndvi'].astype(np.float64).values

        # Independent variables
        X_vars = ['ln_pm25_mean', 'ln_pm25_mean_sq', 'ln_temp_av', 
                  'ln_rain_cum', 'ln_hum_av', 'ln_lag_lumen']

        # # Time & plot fixed effects (sparse format)
        # time_dummy_cols = [col for col in df.columns if col.startswith('time_')]
        # plot_dummy_cols = [col for col in df.columns if col.startswith('plot_')]

        # # Convert fixed effects to sparse matrices
        # X_numeric = df[X_vars].astype(np.float64)
        # X_time_sparse = sparse.csr_matrix(pd.get_dummies(df[time_dummy_cols], drop_first=True))
        # X_plot_sparse = sparse.csr_matrix(pd.get_dummies(df[plot_dummy_cols], drop_first=True))

        # # Combine into a single sparse matrix
        # X_sparse = sparse.hstack([X_numeric, X_time_sparse, X_plot_sparse])
        # X_dense = X_sparse.toarray()
        # X_with_const = sm.add_constant(X_dense)

        # # --- Run OLS ---
        # logger.info("Running OLS regression")
        # ols_model = sm.OLS(y, X_with_const)
        # ols_results = ols_model.fit(cov_type='HC3')
        
        # Add time and plot dummy variables for fixed effects
        time_dummy_cols = [col for col in df.columns if col.startswith('time_')]
        plot_dummy_cols = [col for col in df.columns if col.startswith('plot_')]
        
        # Combine all variables
        all_X_vars = X_vars + time_dummy_cols + plot_dummy_cols
        logger.info(f"Using {len(X_vars)} main variables, {len(time_dummy_cols)} time fixed effects, and {len(plot_dummy_cols)} plot fixed effects")
        
        # Check for non-numeric data in the variables we'll use
        for col in all_X_vars:
            if not pd.api.types.is_numeric_dtype(df[col]):
                logger.warning(f"Column {col} is not numeric. Converting to numeric.")
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Create X matrix for regression - ensure all numeric
        X = df[all_X_vars].astype(np.float64)
        
        # Run OLS with robust standard errors
        logger.info("Fitting OLS model with time and plot fixed effects and robust standard errors")
        X_with_const = sm.add_constant(X)
        ols_model = sm.OLS(y, X_with_const)
        ols_results = ols_model.fit(cov_type='HC3')  # HC3 is a robust covariance estimator

        # Store OLS results
        results = {
            'ols_r2': ols_results.rsquared,
            'ols_adj_r2': ols_results.rsquared_adj,
            'ols_aic': ols_results.aic,
            'pm25_coef': ols_results.params[1],  # First independent variable
            'pm25_sq_coef': ols_results.params[2],
            'pm25_pval': ols_results.pvalues[1],
            'pm25_sq_pval': ols_results.pvalues[2]
        }
        logger.info(f"OLS R²: {results['ols_r2']:.4f}, Adjusted R²: {results['ols_adj_r2']:.4f}")

        # --- SPATIAL REGRESSION ---
        if len(df) > SAMPLE_SIZE:
            logger.info(f"Subsampling {SAMPLE_SIZE} observations for spatial analysis")
            df_sample = df.sample(SAMPLE_SIZE, random_state=42)
        else:
            df_sample = df

        # Spatial coordinates
        coords = np.array(df_sample[['lon', 'lat']].values, dtype=np.float64)
        y_sample = df_sample['ndvi'].values
        X_sample = df_sample[X_vars].astype(np.float64).values
        X_sample = sm.add_constant(X_sample)

        # Create spatial weights matrix
        w = weights.KNN(coords, k=min(8, len(coords) - 1))
        w.transform = 'r'

        def fit_spatial_model(model_class, y, X, w, model_name):
            """Helper function to run spatial regression models."""
            try:
                model = model_class(y, X, w=w, name_y='ndvi', name_x=X_vars)
                r2 = model.r2
                adj_r2 = 1 - (1 - r2) * (len(y) - 1) / (len(y) - X.shape[1] - 1)
                aic = model.aic
                rho_or_lambda = model.rho if hasattr(model, 'rho') else model.lam
                
                pm25_coef = model.betas[1]
                pm25_sq_coef = model.betas[2]

                # Approximate p-values using normal distribution
                pm25_pval = 2 * (1 - stats.norm.cdf(abs(pm25_coef / model.std_err[1])))
                pm25_sq_pval = 2 * (1 - stats.norm.cdf(abs(pm25_sq_coef / model.std_err[2])))

                logger.info(f"{model_name} R²: {r2:.4f}, Adjusted R²: {adj_r2:.4f}")

                return {
                    f'{model_name.lower()}_r2': r2,
                    f'{model_name.lower()}_adj_r2': adj_r2,
                    f'{model_name.lower()}_aic': aic,
                    f'{model_name.lower()}_rho_or_lambda': rho_or_lambda,
                    f'{model_name.lower()}_pm25_coef': pm25_coef,
                    f'{model_name.lower()}_pm25_sq_coef': pm25_sq_coef,
                    f'{model_name.lower()}_pm25_pval': pm25_pval,
                    f'{model_name.lower()}_pm25_sq_pval': pm25_sq_pval
                }

            except Exception as e:
                logger.warning(f"Error in {model_name}: {str(e)}")
                return {}

        # Run spatial models in parallel
        with mp.Pool(processes=2) as pool:
            spatial_results = pool.starmap(
                fit_spatial_model,
                [
                    (ML_Lag, y_sample, X_sample, w, "Spatial Lag"),
                    (ML_Error, y_sample, X_sample, w, "Spatial Error"),
                ],
            )

        # Merge results
        for res in spatial_results:
            results.update(res)

        return results

    except Exception as e:
        logger.error(f"Error processing chunk: {str(e)}")
        return None

### Disabled batch runner

The commented runner would process two Parquet files, remove IDs, and write `final_spatial_regression_results.csv`.


In [ ]:
# start_time = datetime.now()
    
# # Find all Parquet files
# parquet_files = glob.glob(PARQUET_PATH)

# logger.info(f"Found {len(parquet_files)} Parquet files")

# # Process each Parquet file one by one
# all_results = []
# for file in parquet_files[:2]:
#     logger.info(f"Processing {file}")
        
#     # Read parquet file in a memory-efficient way
#     df = dd.read_parquet(file, engine='pyarrow').compute()  # Convert chunk to pandas for processing

#     # Drop missing values
#     required_cols = ['lon', 'lat', 'ndvi'] + [col for col in df.columns if col.startswith('ln_')]
#     df = df.dropna(subset=required_cols)
#     df = df.drop(['time_id', 'plot_id'], axis=1)
        
#     # Process and get results
#     results = process_chunk(df)
#     if results:
#         all_results.append(results)

# # Combine results
# if all_results:
#     final_results = pd.DataFrame(all_results)
#     final_results.to_csv("final_spatial_regression_results.csv", index=False)
#     logger.info("All results saved successfully.")

# end_time = datetime.now()
# logger.info(f"Total execution time: {end_time - start_time}")

## 3. Alternative fits

### Formula OLS

`DataSet` loads requested columns from the full Parquet dataset.
The attempt failed with `MemoryError` for a 356 GiB, 218650 by 218650 array and produced no coefficients.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import statsmodels.formula.api as smf

class DataSet(dict):
    def __init__(self, path):
        self.parquet = pq.ParquetDataset(path)

    def __getitem__(self, key):
        try:
            return self.parquet.read([key]).to_pandas()[key]
        except:
            raise KeyError

df = pd.read_parquet(r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed_optimized\part.0.parquet")
# df.columns = [col.replace('-', '_') for col in df.columns]
# df.columns = [col.replace('.', '_') for col in df.columns]
# df.to_parquet(r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed\part.0_fixed.parquet", index=False)


LargeData = DataSet(r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed_optimized")


time_dummy_cols = [col for col in df.columns if col.startswith('time_')]

y = df['ndvi']

 # Independent variables
X_vars = ['ln_pm25_mean', 'ln_pm25_mean_sq', 'ln_temp_av', 
                  'ln_rain_cum', 'ln_hum_av', 'ln_lag_lumen']

# Add time and plot dummy variables for fixed effects
time_dummy_cols = [col for col in df.columns if col.startswith('time_')]
plot_dummy_cols = [col for col in df.columns if col.startswith('plot_')]

# Combine all variables
all_X_vars = X_vars + time_dummy_cols + plot_dummy_cols
all_X_vars = [var.replace('-', '_') for var in all_X_vars]


equation = "ndvi ~ "
for var in all_X_vars[:]:
    equation += var + ' + '
equation = equation[:-3]


r = smf.ols(equation, data=LargeData).fit(cov_type='HC3')

### Compare dtypes and memory

Reads one partition per branch.
Matching saved dimensions do not prove row equality.


In [ ]:
df = pd.read_parquet(r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed\part.0.parquet")
df2 = pd.read_parquet(r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed_optimized\part.0.parquet")

#### Original dtypes

The saved pandas summary reports 82,637 rows, 1,061 columns, and about 668.9 MB.


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82637 entries, 0 to 82636
Columns: 1061 entries, year to plot_105.358_10.062
dtypes: float64(16), int64(1043), object(2)
memory usage: 668.9+ MB


#### Downcast dtypes

The saved partition reports 82,637 rows, 1,061 columns, and about 88.7 MB.
This measures partition memory only.


In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82637 entries, 0 to 82636
Columns: 1061 entries, year to plot_105.358_10.062
dtypes: float32(16), int32(1), int8(1042), object(2)
memory usage: 88.7+ MB


#### Rounded memory ratio

Divides 669 MB by 89 MB; it does not measure runtime.


In [ ]:
669/89

7.51685393258427

### In-memory OLS

Loads the full optimized dataset into pandas.
The saved attempt failed with `NameError: name 'pd' is not defined`, consistent with a restarted or out-of-order kernel.
Earlier imports supply `pd`; fitting and memory requirements remain unverified.


In [ ]:
df = pd.read_parquet(r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed_optimized")

# df = pd.read_parquet(r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed\part.0.parquet")
# df.columns = [col.replace('-', '_') for col in df.columns]
# df.columns = [col.replace('.', '_') for col in df.columns]

time_dummy_cols = [col for col in df.columns if col.startswith('time_')]

y = df['ndvi']

 # Independent variables
X_vars = ['ln_pm25_mean', 'ln_pm25_mean_sq', 'ln_temp_av', 
                  'ln_rain_cum', 'ln_hum_av', 'ln_lag_lumen']

# Add time and plot dummy variables for fixed effects
time_dummy_cols = [col for col in df.columns if col.startswith('time_')]
plot_dummy_cols = [col for col in df.columns if col.startswith('plot_')]
        
# Combine all variables
all_X_vars = X_vars + time_dummy_cols + plot_dummy_cols
# all_X_vars = [var.replace('-', '_') for var in all_X_vars]
logger.info(f"Using {len(X_vars)} main variables, {len(time_dummy_cols)} time fixed effects, and {len(plot_dummy_cols)} plot fixed effects")
        
# Check for non-numeric data in the variables we'll use
# for col in all_X_vars:
#     if not pd.api.types.is_numeric_dtype(df[col]):
#         logger.warning(f"Column {col} is not numeric. Converting to numeric.")
#         df[col] = pd.to_numeric(df[col], errors='coerce')
        
# Create X matrix for regression - ensure all numeric
X = df[all_X_vars].drop(['time_id', 'plot_id'], axis=1)
        
# Run OLS with robust standard errors
logger.info("Fitting OLS model with time and plot fixed effects and robust standard errors")
X_with_const = sm.add_constant(X)
print("Fitting Model")
ols_model = sm.OLS(y, X_with_const)
ols_results = ols_model.fit(cov_type='HC3')  # HC3 is a robust covariance estimator

### Dask-GLM and covariance

The attempt reported 4,209,948 rows and 1,047 columns, then failed on broadcasting lengths 94,062 and 81,110.
`compute_hc3_se` uses fitted coefficients for leverage instead of the hat-matrix diagonal.
Review its matrix operations and argument types before interpreting any inference.


In [ ]:
import dask.dataframe as dd
import dask.array as da
from dask_glm.estimators import LinearRegression
import dask

# File path to parquet directory
parquet_dir = r"D:\Work\ADB\SAR\datasets\KienGiang_rice_processed_optimized"

# Load dataset from Parquet
df = dd.read_parquet(parquet_dir, engine="pyarrow")
df = df.repartition(npartitions=39)

# Inspect columns
print("Dataset Columns:", df.columns)

# Independent variables
X_vars = ['ln_pm25_mean', 'ln_pm25_mean_sq', 'ln_temp_av', 
          'ln_rain_cum', 'ln_hum_av', 'ln_lag_lumen']

# Add time and plot dummy variables for fixed effects
time_dummy_cols = [col for col in df.columns if col.startswith('time_')]
plot_dummy_cols = [col for col in df.columns if col.startswith('plot_')]

# Combine all variables
all_X_vars = X_vars + time_dummy_cols + plot_dummy_cols

# Ensure time and plot dummy variables exist in the dataset
missing_cols = [col for col in all_X_vars if col not in df.columns]
if missing_cols:
    print(f"Warning: The following columns are missing from the dataset and will be ignored: {missing_cols}")
    all_X_vars = [col for col in all_X_vars if col in df.columns]

# Select X, ensuring 'time_id' and 'plot_id' exist before dropping them
columns_to_drop = [col for col in ['time_id', 'plot_id'] if col in df.columns]
X = df.drop(['year', 'mon', 'lon', 'lat', 'pm25_mean', 'ndvi', 'temp_av', 'hum_av',
       'rain_cum', 'lag_lumen', 'reprod', 'veg', 'time_id', 'plot_id'], axis=1)

# Convert to Dask Arrays
X_da = X.to_dask_array(lengths=True) # More robust than to_dask_array
y_da = df['ndvi'].to_dask_array(lengths=True)  # Convert target to dask array

# Ensure dimensions match
print(f"Shape of X: {X_da.shape}, Shape of y: {y_da.shape}")

# Fit OLS model
ols = LinearRegression(fit_intercept=True)
ols.fit(X_da, y_da)

def compute_hc3_se(X, y, model):
    """
    Compute heteroscedasticity-consistent standard errors (HC3) for a fitted OLS model in Dask.

    Parameters:
    - X (dask.array): Feature matrix
    - y (dask.array): Target vector
    - model (Dask-ML LinearRegression): Trained OLS model

    Returns:
    - HC3 robust standard errors as a dask array
    """
    # Get model predictions
    y_pred = model.predict(X)

    # Ensure y and y_pred have matching shapes
    if y.shape[0] != y_pred.shape[0]:
        raise ValueError(f"Mismatch in dimensions: y has shape {y.shape}, y_pred has shape {y_pred.shape}")

    # Compute residuals
    residuals = y - y_pred

    # Approximate leverage (hat matrix diagonal)
    hat_diag = (X * model.coef_).sum(axis=1) / X.shape[1]

    # HC3 adjustment: Divide residuals by (1 - leverage)^1.5
    weights = da.maximum(1 - hat_diag, 1e-10) ** 1.5  # Avoid division by zero
    robust_residuals = residuals / weights

    # Compute robust variance-covariance matrix
    XTX_inv = da.linalg.inv((X.T @ X) / X.shape[0])  # Approximate inverse(X'X)

    # Compute HC3 robust standard errors
    hc3_variance = (XTX_inv @ (X.T @ (robust_residuals ** 2))) @ XTX_inv
    hc3_se = da.sqrt(da.diag(hc3_variance))

    return hc3_se

# Compute HC3 standard errors
hc3_se = compute_hc3_se(X, y, ols)

# Compute and print results
hc3_se_result = hc3_se.compute()
print("HC3 Robust Standard Errors:", hc3_se_result)


## Remaining model work

Review the specification and check dummies, samples, spatial weights, and covariance before fitting.
Saved outputs support no effect, standard-error, or significance claims.
